# Chapter 12. 모방학습과 인간 피드백 — 12.3 실습: BC vs RL Gridworld

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter12_3_bc_rl_gridworld.ipynb)

책 본문: [12.3 모방학습 vs 강화학습: 언제 무엇을 쓰는가](https://smhanlab.com/book-ml/kor/ml2/chapter12/3.html)

이 노트북은 12.3절의 핵심 주장 — **순수 BC는 전문가 수준에서 멈추고, 순수 RL은
넘을 수 있지만 안전 비용이 크며, BC+RL(하이브리드)은 둘 다를 얻는다** — 를
가장 작은 격자월드에서 정량적으로 확인합니다. 두 부분으로 이뤄집니다:

1. **복합 오차 \(p^T\)**: 매 스텝 99%로 전문가를 따라가도, 긴 episode를
   *완벽하게* 완주할 확률은 지수함수로 떨어져 "매 스텝 잘한다"가
   "긴 episode를 잘 완주한다"를 보장하지 않음을 보여줍니다.
2. **5×5 격자월드, 3개의 팔(arm)**: **순수 BC**, **순수 RL(Q-learning)**,
   **BC+RL(워밍 스타트)**을 같은 환경에서 돌려 *최종 성능*과
   *위험한 행동(함정 진입) 횟수*를 비교합니다.

그림 저장 위치: /home/smhan/book-ml/kor/src/images

In [1]:
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False

import os
import math
import random

import numpy as np

IMG = "/home/smhan/book-ml/kor/src/images"
if not os.path.isdir(IMG):
    IMG = "/tmp"
print("그림 저장 디렉터리:", IMG)

그림 저장 디렉터리: /home/smhan/book-ml/kor/src/images


## 1. 복합 오차 \(p^T\): "매 스��� 잘한다" ≠ "긴 episode를 완주한다"

BC로 잘 학습된 정책이라도 배포 단계에서는 매 스텝마다 *시연에 없던*
미세한 상태 차이·센서 노이즈·환경 교란으로 인해 확률 \(p < 1\)로만
"전문가가 하듯" 올바르게 행동합니다. 매 스텝 독립이라면 \(T\)스텝
episode에서 *한 번도* 오점을 지나치지 않을 확률은 \(p^T\)입니다.
본문의 표와 그래프를 만들어봅니다.

In [2]:
print("=== (a) 복합 오차: P(T 스텝 동안 오점 없음) = p^T ===")
print(f"{'p':<6}{'T=50':>8}{'T=100':>9}{'T=200':>9}{'T=300':>9}")
for p in (0.98, 0.99, 0.995):
    row = "  ".join(f"{p**T:.3f}".rjust(9) for T in (50, 100, 200, 300))
    print(f"{p:<6}{row}")
print()
for p in (0.98, 0.99, 0.995):
    t50 = math.log(0.5) / math.log(p)
    t05 = math.log(0.05) / math.log(p)
    print(f"p={p}: 50% 생존 T={t50:.0f}, 5% 생존 T={t05:.0f}")

=== (a) 복합 오차: P(T 스텝 동안 오점 없음) = p^T ===
p         T=50    T=100    T=200    T=300
0.98      0.364      0.133      0.018      0.002
0.99      0.605      0.366      0.134      0.049
0.995     0.778      0.606      0.367      0.222

p=0.98: 50% 생존 T=34, 5% 생존 T=148
p=0.99: 50% 생존 T=69, 5% 생존 T=298
p=0.995: 50% 생존 T=138, 5% 생존 T=598


In [3]:
T_grid = np.arange(1, 501)
fig, ax = plt.subplots(figsize=(8, 5))
for p, c in ((0.98, "tab:red"), (0.99, "tab:blue"), (0.995, "tab:green")):
    ax.plot(T_grid, p ** T_grid, lw=2, color=c, label=f"per-step success rate p = {p}")
ax.axhline(0.5, color="gray", ls="--", lw=1)
ax.axhline(0.05, color="gray", ls=":", lw=1)
ax.text(12, 0.53, "50%", fontsize=9, color="gray")
ax.text(12, 0.02, "5%", fontsize=9, color="gray")
ax.set_xlabel("Episode length T (number of steps)")
ax.set_ylabel("P(no mistakes in all T steps)  $p^T$")
ax.set_title("Compounding error: even at 99% per-step accuracy,\nthe chance of a flawless long episode is only about 5%")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9, loc="center right")
plt.tight_layout()
fa = os.path.join(IMG, "ch12_3_compounding_error.svg")
plt.savefig(fa, bbox_inches="tight")
plt.close()
print("저장:", fa)

저장: /home/smhan/book-ml/kor/src/images/ch12_3_compounding_error.svg


## 2. 5×5 격자월드: BC, RL, BC+RL 세 가지를 붙여 놓고 달리기

- **환경**: 5×5 격자월드. 시작 (4,0) → 목표 (0,4). 각 스텝마다 보상
  −1이므로 *최단 경로*가 *가장 큰 보상 합*을 줍니다.
- **전문가(시연 제공자)**: **안전하지만 비최적**인 10스텝 우회 경로
  (보상 −10)를 따릅니다. 최적 경로는 8스텝(−8) — *전문가가 최적을
  모르고 가는* 상황입니다.
- **위험**: (2,2)에 "함정". 이 칸에 들어오면 보상 −50, episode 즉시
  종료. 시연 경로에는 함정이 없지만, **5% 확률로** 실제 이동이
  명령과 무관한 인접 칸으로 바뀌는 *교란*이 있어 BC 에이전트가
  *시연 경로 밖*으로 밀려나 *시연에 없던 상태*에서 행동해야 합니다.
- **세 그룹**: (A) **순수 BC** — 시연에서 (s,a)를 배운 정책만,
  시연 밖 상태는 무작위. (B) **순수 RL** — Q-learning을 0에서
  ε-탐험으로 학습. (C) **BC+RL** — BC의 가치함수를 초기값으로
  Q-learning으로 미세 조정.
- 500 episode, seed 0 고정.

In [4]:
S = 5
START, GOAL, PIT = (4, 0), (0, 4), (2, 2)
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # up, down, left, right
DISTURB = 0.05  # 스텝마다: 실제 이동 = 무작위 인접 칸(명령과 무관)
EP_MAX = 60
N_EP = 500

# 전문가 경로: 안전하지만 비최적 (10 스텝; 최적은 8)
EXPERT_PATH = [(4, 0), (3, 0), (3, 1), (2, 1), (1, 1),
               (1, 0), (0, 0), (0, 1), (0, 2), (0, 3), (0, 4)]
assert (2, 2) not in EXPERT_PATH and EXPERT_PATH[-1] == GOAL
expert_action = {}
for s, ns in zip(EXPERT_PATH, EXPERT_PATH[1:]):
    d = (ns[0] - s[0], ns[1] - s[1])
    expert_action[s] = ACTIONS.index(d)
print(f"전문가 경로 길이 = {len(EXPERT_PATH) - 1} (보상 -{len(EXPERT_PATH)-1}); 최적 = 8")

def neighbors(s):
    r, c = s
    return [(min(max(r + ACTIONS[a][0], 0), S - 1),
             min(max(c + ACTIONS[a][1], 0), S - 1)) for a in range(4)]

def step(s, a, rng):
    if rng.random() < DISTURB:
        ns = rng.choice(neighbors(s))      # 교란: 무작위로 인접 칸으로 밀림
    else:
        r, c = s
        ns = (min(max(r + ACTIONS[a][0], 0), S - 1),
              min(max(c + ACTIONS[a][1], 0), S - 1))
    if ns == PIT:
        return ns, -50.0, True
    return ns, -1.0, (ns == GOAL)

전문가 경로 길이 = 10 (보상 -10); 최적 = 8


In [5]:
# --- 팔 A: 순수 행동복제(깨끗한 시연, 결정론적)
bc_policy = dict(expert_action)

def run_bc(n_ep, seed):
    rng = random.Random(seed)
    rets, pit_at = [], []
    for ep in range(n_ep):
        s, ret, done = START, 0.0, False
        for _t in range(EP_MAX):
            a = bc_policy[s] if s in bc_policy else rng.randrange(4)  # 시연 밖 -> 무작위
            ns, r, done = step(s, a, rng)
            ret += r
            if ns == PIT:
                pit_at.append(ep + 1)
            s = ns
            if done:
                break
        rets.append(ret)
    return rets, pit_at

# --- Q-learning 헬퍼 (팔 B, C)
states_all = [(r, c) for r in range(S) for c in range(S) if (r, c) != PIT]

def q_learning(n_ep, q0, seed, eps0=0.5, eps_min=0.05, alpha=0.1, gamma=0.95):
    rng = random.Random(seed)
    rets, pit_at = [], []
    for ep in range(n_ep):
        eps = max(eps_min, eps0 * 0.97 ** ep)
        s, ret, done = START, 0.0, False
        for _t in range(EP_MAX):
            a = rng.randrange(4) if rng.random() < eps else max(range(4), key=lambda x: q0[s][x])
            q_old = q0[s][a]
            ns, r, done = step(s, a, rng)
            target = r if done else r + gamma * max(q0[ns])
            q0[s][a] = q_old + alpha * (target - q_old)
            ret += r
            if ns == PIT:
                pit_at.append(ep + 1)
            s = ns
            if done:
                break
        rets.append(ret)
    return rets, pit_at

ret_bc, pit_bc = run_bc(N_EP, seed=0)
q_scratch = {s: [0.0] * 4 for s in states_all}
ret_rl, pit_rl = q_learning(N_EP, q_scratch, seed=0, eps0=0.5)
# 워밍 스타트 = "BC가 정확한 초기 가치함수를 준다":
# 시연 경로 위 Q(s, 전문가행동) = -남은 스텝 수(실제 cost-to-go),
# 다른 행동 = 1스텝 나쁨, 경로 밖 상태는 0(아직 미발견, RL이 채움)
q_warm = {s: [0.0] * 4 for s in states_all}
for i, s in enumerate(EXPERT_PATH[:-1]):
    remaining = (len(EXPERT_PATH) - 1) - i   # 10, 9, ..., 1
    for a in range(4):
        q_warm[s][a] = -remaining if a == expert_action[s] else -remaining + 1.0
ret_warm, pit_warm = q_learning(N_EP, q_warm, seed=0, eps0=0.2)
print("세 그룹 실행 완료 (각 500 episode).")

세 그룹 실행 완료 (각 500 episode).


## 3. 결과: 최종 성능과 안전 비용

- **최종 성능** = 마지막 20 episode의 평균 보상(20-episode 이동평균).
  높을수록, −8(= 최적)에 가까울수록 좋습니다.
- **"전문가 추월" episode** = 20-episode 이동평균이 전문가 수준(−10)을
  처음 넘는 episode. *없으면* 전문가를 못 넘은 것입니다.
- **안전 비용** = 500 episode 동안 위험 칸(함정)에 진입한 총 횟수.

In [6]:
def moving_avg(x, w=20):
    return [sum(x[max(0, i - w + 1):i + 1]) / min(i + 1, w) for i in range(len(x))]

def first_beat_expert(rets, thr=-9.5, w=20):
    """w-episode 이동평균이 전문가 수준을 처음 넘는 episode"""
    m = moving_avg(rets, w)
    for i, v in enumerate(m):
        if i + 1 >= w and v > thr:
            return i + 1
    return None

arms = {
    "순수 BC": (ret_bc, pit_bc),
    "순수 RL": (ret_rl, pit_rl),
    "BC+RL": (ret_warm, pit_warm),
}
print(f"{'그룹':<10}{'마지막20평균':>12}{'함정진입':>10}{'전문가추월ep':>13}{'초기50ep함정':>13}")
for name, (rets, pits) in arms.items():
    b = first_beat_expert(rets)
    early = sum(1 for e in pits if e <= 50)
    print(f"{name:<10}{np.mean(rets[-20:]):>12.2f}{len(pits):>10}/{N_EP}"
          f"{str(b):>13}{early:>13}")

그룹             마지막20평균      함정진입      전문가추월ep     초기50ep함정
순수 BC           -10.65        26/500         None            1
순수 RL            -8.70        30/500          273           20
BC+RL            -9.15         9/500          193            6


In [7]:
x = np.arange(1, N_EP + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.5))
ax1.plot(x, moving_avg(ret_bc), color="tab:gray", lw=1.8, label="Pure BC (expert demos only)")
ax1.plot(x, moving_avg(ret_rl), color="tab:red", lw=1.4, alpha=0.9, label="Pure RL (starting from 0)")
ax1.plot(x, moving_avg(ret_warm), color="tab:blue", lw=1.8, label="BC+RL (warm start)")
ax1.axhline(-8, color="green", ls="--", lw=1, label="Optimal path = -8 (no disturbance)")
ax1.set_xlabel("episode")
ax1.set_ylabel("Mean reward (20-ep moving average)")
ax1.set_title("(a) Final performance: RL reaches the 8-step optimum\n(beating the expert's 10-step detour), while pure BC plateaus")
ax1.legend(fontsize=8, loc="center right")

names = list(arms.keys())
cnt = [len(arms[n][1]) for n in names]
bars = ax2.bar(names, cnt, color=["tab:gray", "tab:red", "tab:blue"], alpha=0.85)
for b, v in zip(bars, cnt):
    ax2.text(b.get_x() + b.get_width() / 2, v + max(cnt) * 0.01, str(v),
             ha="center", fontsize=9)
ax2.set_ylabel("Pitfalls entered (risky cell, reward -50)\nover 500 episodes")
ax2.set_title("(b) Safety cost: number of risky actions\nattempted in the environment during learning")
plt.tight_layout()
fb = os.path.join(IMG, "ch12_3_bc_rl_comparison.svg")
plt.savefig(fb, bbox_inches="tight")
plt.close()
print("저장:", fb)

저장: /home/smhan/book-ml/kor/src/images/ch12_3_bc_rl_comparison.svg


## 4. 12.3절의 어떤 주장을 확인해주는가

- **순수 BC**: 전문가의 10스텝 수준(≈ −10.6)에서 멈추고, *단 한 번도*
  전문가를 넘지 못함 — "시연은 상한이 사람"이라는 12.2절의 주장.
- **순수 RL**: 전문가를 넘어 8스텝 최적에 도달하지만, 학습 중(특히
  초기) **위험한 행동을 가장 많이** 시도함(함정 진입이 가장 많고,
  초기 50 episode에 집중).
- **BC+RL**: 순수 RL보다 *빠르게* 전문가를 추월하고, **위험한 행동이
  가장 적음** — "실전에서는 둘을 섞어 쓴다"의 정량적 근거.

> 세 그룹의 *순위*(BC는 상한 / RL은 초과하지만 위험 / 하이브리드는
> 안전+빠름)는 랜덤 시드에 관계없이 견고합니다. 구체적인 숫자
> (마지막-20 평균, 함정 횟수, 추월 episode)는 시드에 따라 다소
> 흔들릴 수 있습니다.